# Aerial Object Adversarial Evaluation Workflow

In this exercise, you will evaluate a binary aerial object classifier under adversarial evasion attacks. The positive class is CIFAR-10 `airplane`, which stands in for an unauthorized aerial object in a restricted-airspace monitoring scenario.

## 1. Imports and Environment Setup

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

EXERCISE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("module-7-apply-evasion-attacks/exercise/starter")
ART_HOME = EXERCISE_ROOT / "art_data"
ART_HOME.mkdir(parents=True, exist_ok=True)
os.environ["ART_DATA_PATH"] = str(ART_HOME)
os.environ["HOME"] = str(EXERCISE_ROOT)
os.environ["USERPROFILE"] = str(EXERCISE_ROOT)
sys.path.append(str(EXERCISE_ROOT / "src"))

from aerial_eval_utils import (
    CLASS_NAMES,
    attack_success_rate,
    build_art_classifier,
    evaluate_numpy,
    load_aerial_dataset,
    make_attack,
    perturbation_metrics,
    plot_comparison_grid,
    predict,
    save_metrics_table,
    train_or_load_model,
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
else:
    device = "cpu"
print(f"Using device: {device}")

## 2. Load the Provided Model and Validation Dataset

Run `bash scripts/prepare_airplane_assets.sh` from the exercise folder before this step. On Windows without Bash, run `python scripts/prepare_airplane_assets.py`.

In [ ]:
data_dir = EXERCISE_ROOT / "data/generated"
model_path = EXERCISE_ROOT / "models/aerial_object_cnn.pt"

model = train_or_load_model(model_path, data_dir, device=device, force_train=False)
val_dataset = load_aerial_dataset(data_dir, split="val")

val_images = val_dataset.images
val_labels = val_dataset.labels

print(f"Validation images: {val_images.shape}")
print(f"Class names: {CLASS_NAMES}")

## 3. Measure Clean Baseline Performance

In [ ]:
clean_metrics = evaluate_numpy(model, val_images, val_labels)
clean_predictions = clean_metrics["predictions"]
clean_confidence = clean_metrics["confidence"]

print(f"Clean accuracy: {clean_metrics['accuracy']:.3f}")
print(f"Mean clean confidence: {clean_metrics['mean_confidence']:.3f}")
print(f"Clean false negative rate: {clean_metrics['false_negative_rate']:.3f}")

## 4. Select One Evasion Attack

Choose one attack from: `fgsm`, `pgd`, `bim`, `deepfool`, or `simba`.

TODO: Set `attack_name` and at least three epsilon values. Start small and increase gradually.

In [ ]:
attack_name = "TODO"  # TODO: choose fgsm, pgd, bim, deepfool, or simba
epsilon_values = [0.0]  # TODO: replace with at least three values, for example [0.01, 0.03, 0.06]

if attack_name == "TODO" or len(epsilon_values) < 3:
    raise ValueError("Choose an attack and provide at least three epsilon values before continuing.")

classifier = build_art_classifier(model)
print(f"Selected attack: {attack_name}")
print(f"Epsilon values: {epsilon_values}")

## 5. Generate Adversarial Examples and Compute Metrics

TODO: Complete the loop so each epsilon generates adversarial examples and computes adversarial accuracy, attack success rate, confidence degradation, and perturbation metrics.

In [ ]:
evaluation_images = val_images[:120]
evaluation_labels = val_labels[:120]
clean_eval = evaluate_numpy(model, evaluation_images, evaluation_labels)

rows = []
adversarial_examples_by_epsilon = {}

for epsilon in epsilon_values:
    # TODO: create the attack with make_attack(...)
    # TODO: generate adversarial examples with attack.generate(...)
    # TODO: evaluate adversarial examples with evaluate_numpy(...)
    # TODO: compute perturbation metrics with perturbation_metrics(...)
    # TODO: append a row dictionary using the keys in docs/metrics_template.md
    raise NotImplementedError("Complete the adversarial evaluation loop.")

metrics_path = save_metrics_table(rows, EXERCISE_ROOT / "results/adversarial_metrics.csv")
print(f"Saved metrics to: {metrics_path}")
rows

## 6. Visualize Clean and Adversarial Predictions

TODO: Pick one epsilon from your run and create a comparison grid.

In [ ]:
selected_epsilon = epsilon_values[-1]
selected_adv = adversarial_examples_by_epsilon[selected_epsilon]
adv_predictions, adv_confidence, _ = predict(model, selected_adv)

comparison_path = plot_comparison_grid(
    evaluation_images,
    selected_adv,
    clean_eval["predictions"],
    adv_predictions,
    evaluation_labels,
    EXERCISE_ROOT / f"results/{attack_name}_epsilon_{selected_epsilon}_comparison.png",
)
print(f"Saved comparison grid to: {comparison_path}")

## 7. Write the Robustness Assessment

Update `docs/robustness_assessment.md` with your quantitative findings and operational risk discussion. Your answer should mention the most risky epsilon setting, attack success rate, adversarial accuracy, confidence degradation, and false-negative impact.